# Autoregressive Evaluation and Figures


In [ ]:
if "config" not in globals():
    raise RuntimeError("Run 00_experiment_runner.ipynb first so config is defined.")

ar_results = {}
metric_rows = []
metrics_df = None


## Autoregressive Prediction
## Metrics


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error


def compute_basic_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1)))),
        "MAE": float(mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))),
    }


@torch.no_grad()
def autoregressive_predict_joint(
    model,
    xb,
    total_output_len,
    base_len,
):
    """
    자유 AR rollout.

    xb:
        모델 입력 공간, shape (B, input_len, input_dim)

    model output:
        scaled target 공간, shape (B, base_len, target_dim)

    동작:
        예측한 heave와 surge를 모두 다음 pre-PCA 입력에 삽입한 후
        PCA transform하여 다음 모델 입력을 생성합니다.
    """
    if total_output_len <= 0:
        raise ValueError("total_output_len은 양수여야 합니다.")
    if total_output_len % base_len != 0:
        raise ValueError("현재 설정에서는 total_output_len이 base_len의 배수여야 합니다.")

    model.eval()
    model_device = next(model.parameters()).device
    current_model_np = xb.detach().cpu().numpy()
    current_pre_np = model_to_pre_pca_scaled(current_model_np)

    predictions = []
    generated = 0

    while generated < total_output_len:
        model_input_np = pre_pca_to_model_scaled(current_pre_np)
        model_input = torch.tensor(
            model_input_np,
            dtype=torch.float32,
            device=model_device,
        )

        pred = model(model_input)
        if pred.ndim != 3:
            raise RuntimeError(f"모델 출력은 (B,T,F)여야 합니다: {pred.shape}")
        if pred.shape[-1] != len(target_cols):
            raise RuntimeError(
                f"target_dim 불일치: model={pred.shape[-1]}, expected={len(target_cols)}"
            )

        step_len = min(base_len, total_output_len - generated)
        pred_chunk = pred[:, :step_len, :]
        predictions.append(pred_chunk)
        pred_np = pred_chunk.detach().cpu().numpy()

        # 입력에만 있고 출력하지 않는 보조 feature가 있다면 마지막 값을 유지합니다.
        next_pre = np.repeat(current_pre_np[:, -1:, :], step_len, axis=1)

        # heave/surge 등 target으로 예측한 변수는 모두 새 예측값으로 갱신합니다.
        for target_idx, col in enumerate(target_cols):
            if col not in pre_pca_input_cols:
                raise KeyError(f"AR 입력에서 target 변수를 찾을 수 없습니다: {col}")
            input_idx = pre_pca_input_cols.index(col)
            next_pre[:, :, input_idx] = pred_np[:, :, target_idx]

        current_pre_np = np.concatenate(
            [current_pre_np[:, step_len:, :], next_pre],
            axis=1,
        )
        generated += step_len

    return torch.cat(predictions, dim=1)


# ============================================================
# 전체 Test set 200-step 자유 AR 평가
# ============================================================
ar_results = {}
heave_idx = target_cols.index(report_col)

for model_name, result_data in results.items():
    model = result_data["model"].to(device)
    pred_batches = []
    true_batches = []

    for xb, yb in test_loader:
        xb = xb.to(device)

        pred_scaled = autoregressive_predict_joint(
            model=model,
            xb=xb,
            total_output_len=EVAL_OUTPUT_LEN,
            base_len=MODEL_OUTPUT_LEN,
        )

        pred_batches.append(pred_scaled.cpu().numpy())
        true_batches.append(yb[:, :EVAL_OUTPUT_LEN, :].cpu().numpy())

    preds_scaled_all = np.concatenate(pred_batches, axis=0)
    trues_scaled_all = np.concatenate(true_batches, axis=0)

    preds_orig_all = inverse_scale_targets(preds_scaled_all)
    trues_orig_all = inverse_scale_targets(trues_scaled_all)

    preds_heave = preds_orig_all[..., heave_idx]
    trues_heave = trues_orig_all[..., heave_idx]

    ar_results[model_name] = {
        "preds_scaled_all": preds_scaled_all,
        "trues_scaled_all": trues_scaled_all,
        "preds_orig_all": preds_orig_all,
        "trues_orig_all": trues_orig_all,
        # 기존 시각화/샘플 코드가 바로 사용할 heave-only 배열
        "preds_orig": preds_heave,
        "trues_orig": trues_heave,
    }

    print(
        f"{model_name}: scaled={preds_scaled_all.shape}, "
        f"original={preds_orig_all.shape}, heave={preds_heave.shape}"
    )

# ============================================================
# 100-step 직접예측 구간 / 200-step AR 누적 구간 지표
# ============================================================
prediction_lengths = [100, 200]
detailed_metrics = []

for model_name, result_data in ar_results.items():
    preds_heave = result_data["preds_orig"]
    trues_heave = result_data["trues_orig"]
    row = {"Model": model_name}

    for length in prediction_lengths:
        metrics = compute_basic_metrics(
            trues_heave[:, :length],
            preds_heave[:, :length],
        )
        sample_rmse = np.sqrt(
            np.mean(
                (trues_heave[:, :length] - preds_heave[:, :length]) ** 2,
                axis=1,
            )
        )
        row[f"RMSE@{length}"] = metrics["RMSE"]
        row[f"RMSE_STD@{length}"] = float(np.std(sample_rmse))
        row[f"MAE@{length}"] = metrics["MAE"]

    detailed_metrics.append(row)

detailed_metrics_df = pd.DataFrame(detailed_metrics)
print("\n모델별 heave Test 성능:")
display(detailed_metrics_df)

metrics_path = os.path.join(
    config["experiment_dir"],
    "test_heave_metrics_ar200.csv",
)
detailed_metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
print("지표 저장:", metrics_path)


## Representative Sample Selection


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# ============================================================
# 샘플 1개에 대해 모델별 200-step AR 예측 생성
# ============================================================
sample_idx = 0
xb, yb = test_dataset[sample_idx]

x_pre_scaled = model_to_pre_pca_scaled(xb.cpu().numpy())
report_input_idx = pre_pca_input_cols.index(report_col)
x_input_orig = inverse_scale_feature(
    x_pre_scaled[:, report_input_idx],
    report_col,
).flatten()

y_true_scaled_all = yb.cpu().numpy()
y_true_orig_all = inverse_scale_targets(y_true_scaled_all)
y_true_orig = y_true_orig_all[:, heave_idx]

xb_model = xb.unsqueeze(0).to(device)
pred_dict = {}
pred_state_dict = {}

MODEL_COLORS = {
    "RNN": "slategray",
    "LSTM": "mediumpurple",
    "BiLSTM": "coral",
    "TCN": "mediumseagreen",
    "Transformer": "royalblue",
}

model_colors = MODEL_COLORS

for model_name, result in results.items():
    pred_scaled = autoregressive_predict_joint(
        model=result["model"],
        xb=xb_model,
        total_output_len=EVAL_OUTPUT_LEN,
        base_len=MODEL_OUTPUT_LEN,
    ).cpu().numpy()[0]

    pred_orig_all = inverse_scale_targets(pred_scaled)
    pred_state_dict[model_name] = pred_orig_all
    pred_dict[model_name] = pred_orig_all[:, heave_idx]

print("샘플 예측 완료")
print("input heave:", x_input_orig.shape)
print("true states:", y_true_orig_all.shape)
for model_name, pred in pred_state_dict.items():
    print(model_name, pred.shape)


In [ ]:
import numpy as np

sample_rmse_stats_dict = {}
metric_output_len = int(config.get("metric_output_len", 200))

for model_name, data in ar_results.items():
    preds_heave = data["preds_orig"][:, :metric_output_len]
    trues_heave = data["trues_orig"][:, :metric_output_len]

    sample_rmse_array = np.sqrt(
        np.mean((preds_heave - trues_heave) ** 2, axis=1)
    )
    sample_rmse_stats_dict[model_name] = {
        metric_output_len: {"sample_rmse_array": sample_rmse_array}
    }

available_models = [m for m in config["model_names"] if m in sample_rmse_stats_dict]
if not available_models:
    raise ValueError("config[\"model_names\"]에 해당하는 AR 결과가 없습니다.")

rmse_arrays = {
    model_name: sample_rmse_stats_dict[model_name][metric_output_len]["sample_rmse_array"]
    for model_name in available_models
}
combined_score = np.mean(list(rmse_arrays.values()), axis=0)
best_idx_combined = int(np.argmin(combined_score))
best_idx_for_figures = np.argsort(combined_score)[:5]

print("평가에 포함된 모델:", available_models)
print("모든 선택 모델을 종합해 잘 맞춘 샘플 idx:", best_idx_combined)
for model_name, rmse_array in rmse_arrays.items():
    print(model_name, "RMSE:", rmse_array[best_idx_for_figures])


In [ ]:
sorted_idx = best_idx_for_figures

print(f"\n평가 길이: {metric_output_len} step")
for rank, idx in enumerate(sorted_idx, start=1):
    model_scores = " | ".join(
        f"{model_name} RMSE={rmse_arrays[model_name][idx]:.6f}"
        for model_name in available_models
    )
    print(f"{rank}위 | sample {idx} | {model_scores} | 합계={combined_score[idx]:.6f}")


## Prediction Figures


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Y_LIM = None  # 예: (-3, 3). None이면 자동 범위

for sample_idx in best_idx_for_figures:
    xb, yb = test_dataset[int(sample_idx)]

    x_pre_scaled = model_to_pre_pca_scaled(xb.cpu().numpy())
    x_input_heave = inverse_scale_feature(
        x_pre_scaled[:, pre_pca_input_cols.index(report_col)],
        report_col,
    ).flatten()

    y_true_all = inverse_scale_targets(yb.cpu().numpy())
    y_true_heave = y_true_all[:, heave_idx]

    sample_idx = int(sample_idx)

    model_names_to_plot = [
        m for m in config["model_names"]
        if m in ar_results
    ]

    # RMSE 순위 계산에 사용한 예측값 그대로 사용
    sample_predictions = {
        model_name: np.asarray(
            ar_results[model_name]["preds_orig"][
                sample_idx, :metric_output_len
            ]
        ).reshape(-1)
        for model_name in model_names_to_plot
    }

    # 정답도 동일한 ar_results에서 가져오기
    y_true_heave = np.asarray(
        ar_results[model_names_to_plot[0]]["trues_orig"][
            sample_idx, :metric_output_len
        ]
    ).reshape(-1)

    model_names_to_plot = list(sample_predictions)
    fig, axes = plt.subplots(
        1,
        len(model_names_to_plot),
        figsize=(7.5 * len(model_names_to_plot), 5),
        sharey=True,
    )
    axes = np.atleast_1d(axes)

    x_input_axis = np.arange(-INPUT_LEN, 0)
    x_future_axis = np.arange(0, EVAL_OUTPUT_LEN)
    
    for ax, model_name in zip(axes, model_names_to_plot):
        ax.plot(x_input_axis, x_input_heave, color="gray", linewidth=2, label="Input heave")
        ax.plot(x_future_axis, y_true_heave, color="black", linestyle="--", linewidth=2, label="True heave")
        ax.plot(
            x_future_axis,
            sample_predictions[model_name],
            color=model_colors.get(model_name, "tab:gray"),
            linewidth=2,
            label=f"{model_name} prediction",
        )
        ax.axvline(0, color="black", linestyle=":", alpha=0.8)
        if Y_LIM is not None:
            ax.set_ylim(Y_LIM)
        ax.set_title(f"{model_name} | sample {sample_idx}")
        ax.set_xlabel("Time step")
        ax.set_ylabel("Heave")
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend(fontsize=8)

    plt.suptitle(f"200-step free AR | {config['experiment_name']}", fontweight="bold")
    plt.tight_layout()
    plt.show()


## Final Summary


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams.update({
    "font.family": "Times New Roman",  # 또는 "Arial"
    "font.size": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "axes.unicode_minus": False,
})

PLOT_OUTPUT_LEN = 200
SAVE_DIR = "./paper_figures"
os.makedirs(SAVE_DIR, exist_ok=True)

# config에 지정된 모델을 모두 시각화합니다.
model_names_to_plot = [model_name for model_name in config["model_names"] if model_name in results]
if not model_names_to_plot:
    raise ValueError("config[\"model_names\"]에 해당하는 학습 결과가 없습니다.")
MODEL_COLORS = {
    "RNN": "slategray",
    "LSTM": "mediumpurple",
    "BiLSTM": "coral",
    "TCN": "mediumseagreen",
    "Transformer": "royalblue",
}

model_colors = {name: MODEL_COLORS.get(name, "tab:gray") for name in model_names_to_plot}

sample_idx = int(best_idx_combined)
xb, yb = test_dataset[sample_idx]

# 입력 heave 복원
x_pre_scaled = model_to_pre_pca_scaled(xb.cpu().numpy())
x_input_orig = inverse_scale_feature(
    x_pre_scaled[:, pre_pca_input_cols.index(report_col)],
    report_col,
).flatten()

# 정답 heave 복원
y_true_all = inverse_scale_targets(yb.cpu().numpy())
y_true_heave = y_true_all[:, heave_idx]

xb_model = xb.unsqueeze(0).to(device)

# 각 모델 예측 저장
pred_dict = {}

for model_name in model_names_to_plot:
    result = results[model_name]

    model = result["model"]
    model.eval()

    with torch.no_grad():
        pred_scaled = autoregressive_predict_joint(
            model=model,
            xb=xb_model,
            total_output_len=PLOT_OUTPUT_LEN,
            base_len=MODEL_OUTPUT_LEN,
        ).cpu().numpy()[0]

    pred_heave = inverse_scale_targets(pred_scaled)[:, heave_idx]
    pred_dict[model_name] = pred_heave

    # -----------------------------
    # 1) 개별 그래프 저장
    # -----------------------------
    plt.figure(figsize=(9, 5))
    plt.plot(
        np.arange(-INPUT_LEN, 0),
        x_input_orig,
        color="black",
        linestyle="--",
        linewidth=2.2,
        label="Input heave"
    )
    plt.plot(
        np.arange(PLOT_OUTPUT_LEN),
        y_true_heave[:PLOT_OUTPUT_LEN],
        color="black",
        linewidth=2.2,
        label="Ground truth"
    )
    plt.plot(
        np.arange(PLOT_OUTPUT_LEN),
        pred_heave[:PLOT_OUTPUT_LEN],
        color=model_colors[model_name],
        linewidth=2.0,
        label=f"{model_name} prediction"
    )

    plt.xlabel("Time step")
    plt.ylabel("Heave")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout()

    file_name = f"{SAVE_DIR}/{model_name.lower()}_sample_{sample_idx}_200step.png"
    plt.savefig(file_name, dpi=300, bbox_inches="tight", facecolor="white")
    print("저장 완료:", file_name)
    plt.show()
    plt.close()

# -----------------------------
# 2) 합친 비교 그래프 저장
# -----------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    np.arange(-INPUT_LEN, 0),
    x_input_orig,
    color="black",
    linestyle="--",
    linewidth=2.2,
    label="Input heave"
)
plt.plot(
    np.arange(PLOT_OUTPUT_LEN),
    y_true_heave[:PLOT_OUTPUT_LEN],
    color="black",
    linewidth=2.2,
    label="Ground truth"
)

for model_name in model_names_to_plot:
    if model_name not in pred_dict:
        continue
    plt.plot(
        np.arange(PLOT_OUTPUT_LEN),
        pred_dict[model_name][:PLOT_OUTPUT_LEN],
        color=model_colors[model_name],
        linewidth=2.0,
        label=f"{model_name} prediction"
    )

plt.xlabel("Time step")
plt.ylabel("Heave")
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()

combined_file_name = f"{SAVE_DIR}/combined_sample_{sample_idx}_200step.png"
plt.savefig(combined_file_name, dpi=300, bbox_inches="tight", facecolor="white")
print("저장 완료:", combined_file_name)
plt.show()
plt.close()